# W06A A · 가정과 잔차

2026-10-05 · **R 커널** · 런타임→모두 실행. A/B는 각각 독립 실행됩니다. 저장된 출력과 현재 런타임의 객체는 다릅니다.

**먼저 예상→실행→설명** 순서로 진행합니다. R/QMD/webR와 Colab은 대체 환경이므로 한 경로만 선택하세요. 예제 코드는 바로 실행되며 연습 셀은 주석으로 되어 있어 기본 흐름을 막지 않습니다. 연습 때 `#`를 지우고 완성하세요.

데이터 역할을 구별하세요: Anscombe/Hamilton은 설명용 구성 자료, attitude는30개 부서의 실제 설문 집계, 잔차 패턴은 조건을 정한 모의 자료입니다. 수업 질문은 `주어진 회귀식에 무엇이 남았는가?`입니다.

공통 코드 고정 버전: **2026-fall-w06a**. [함수 입력·출력·정의 안내](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics-api.md). `source()`는 정의를 읽고, `lm()`은 모형을 적합하며, 진단 함수는 그 결과를 읽습니다. 캐시 재사용 때도 파일 동일성을 확인합니다. 런타임 재시작 후 준비 셀부터 다시 실행하세요.

In [ ]:
# 새 R 런타임에서도 단독 실행됩니다. API 키/수동 업로드가 필요 없습니다.
tag <- "2026-fall-w06a"
local_dirs <- c("labs/student/w06a/data","../../labs/student/w06a/data")
found <- local_dirs[file.exists(file.path(local_dirs,"diagnostics.R"))]
data_dir <- if(length(found)) found[1] else file.path(path.expand("~"),".cache","regre",tag)
dir.create(data_dir,recursive=TRUE,showWarnings=FALSE)
expected <- c("diagnostics.R"="1d6d198179eb52922e0b7da3a3836838","hamilton.csv"="9dfd652f3a9efb8d751961d609bca459")
paths <- c("diagnostics.R"="src/diagnostics.R","hamilton.csv"="data/sample/hamilton.csv")
options(timeout=120)
for(name in names(paths)) {
  target <- file.path(data_dir,name)
  if(!file.exists(target)) {
    url <- paste0("https://raw.githubusercontent.com/lunalab-ai/regre/",tag,"/",paths[[name]])
    status <- download.file(url,target,mode="wb")
    if(status!=0) stop("다운로드 실패: 인터넷을 확인하고 준비 셀을 재실행하세요.")
    cat(name,": 고정 버전 최초 다운로드\n")
  } else cat(name,": 동봉 파일 또는 캐시 확인\n")
  if(unname(tools::md5sum(target))!=expected[[name]]) stop("파일 동일성 확인 실패. 수정 파일은 보관하고 고정 버전으로 다시 시작하세요.")
}
source(file.path(data_dir,"diagnostics.R"))
d <- datasets::attitude
fit <- lm(rating~complaints+learning,data=d)
res <- diag_residuals(fit)
ham <- diag_hamilton(file.path(data_dir,"hamilton.csv"))
cat(R.version.string,"\n30 departments; Hamilton: 15 constructed observations\n")

## 1. 숫자가 같으면 같은 자료일까요?

Anscombe는 설명용 구성 자료입니다. 네 자료의 계수와 R²는 반올림 수준에서 비슷합니다. 그림을 보기 전에 예상하고, 같은 축에서 비교하세요.

In [ ]:
ans_summary <- do.call(rbind,lapply(1:4,function(j) {
  f <- lm(anscombe[[j+4]]~anscombe[[j]])
  data.frame(set=j,intercept=unname(coef(f)[1]),slope=unname(coef(f)[2]),R2=summary(f)$r.squared)
}))
ans_summary
op <- par(mfrow=c(2,2),mar=c(4,4,2,1))
for(j in 1:4) {
  plot(anscombe[[j]],anscombe[[j+4]],xlim=c(3,20),ylim=c(2,14),pch=19,
       col="#087f8c",xlab="X",ylab="Y",main=paste("Anscombe",j))
  abline(lm(anscombe[[j+4]]~anscombe[[j]]),col="#d95d52",lwd=2)
}
par(op)

### L1 · 예측–실행–설명

네 그림 중 곡선 관계가 보이는 자료를 고르고, 요약표만으로 이를 알 수 없는 이유를 두 문장으로 쓰세요.

In [ ]:
# selected_set <- ...
# 근거: 그림에서 관측되는 패턴과 직선의 차이를 적으세요.

## 2. 한 부서의 잔차를 직접 만들기

[diag_residuals 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics.R#L16)는 lm을 받아 관측값·적합값·보통잔차·지레값·두 표준화잔차 표를 반환합니다. rating과 보통잔차 단위는 각각 %와 %p이고, 표준화잔차는 단위가 없습니다.

In [ ]:
res[1,c("row","observed","fitted","raw","h","s","internal","external")]
i <- 1L
e_i <- d$rating[i]-fitted(fit)[i]
denominator <- sigma(fit)*sqrt(1-hatvalues(fit)[i])
c(raw=e_i,denominator=denominator,internal=e_i/denominator)
plot(res$fitted,res$observed,pch=19,col="#087f8c",xlab="Fitted rating (%)",ylab="Observed rating (%)")
abline(0,1,lty=2)
segments(res$fitted[i],res$fitted[i],res$fitted[i],res$observed[i],col="#d95d52",lwd=4)
points(res$fitted[i],res$observed[i],pch=19,col="#d95d52",cex=1.5)

### L2 · 예측–실행–설명

`e_i/sigma(fit)`를 교재의 내적 표준화잔차라고 적은 코드를 고치세요. 빠진 항이 무엇인지 말하세요.

In [ ]:
# corrected <- e_i / (...)
# all.equal(unname(corrected),unname(rstandard(fit)[i]))

## 3. 외적 표준화: 분모만 밖에서 추정하기

`rstudent()`는 교재의 외적 표준화잔차입니다. 관측 i를 제외해 오차 규모를 추정하되 분자는 전체 자료에서 구한 e_i입니다. 아래에서는 관측을 실제로 제외한 적합과 분산 항등식을 대조합니다.

In [ ]:
nu <- df.residual(fit)
s_deleted_formula <- sqrt((deviance(fit)-res$raw[i]^2/(1-res$h[i]))/(nu-1))
fit_without_i <- lm(rating~complaints+learning,data=d[-i,])
c(formula=s_deleted_formula,refit=sigma(fit_without_i))
c(internal=rstandard(fit)[i],external=rstudent(fit)[i])
predictive_deleted <- d$rating[i]-predict(fit_without_i,newdata=d[i,,drop=FALSE])
c(original_e=res$raw[i],deleted_prediction_residual=unname(predictive_deleted))

### L3 · 예측–실행–설명

외적 분자에 삭제예측잔차를 넣으면 값이 바뀝니다. 원래 e_i로 외적 표준화잔차를 완성하고 R과 확인하세요.

In [ ]:
# external_manual <- ... / (s_deleted_formula*sqrt(1-res$h[i]))

## 4. 잔차합 0은 건강 진단 결과일까요?

절편 포함 최소제곱은 잔차합을0으로 만드는 직선을 선택합니다. 아래 곡선 자료도 같은 대수적 성질을 가집니다. 모의 자료이며 실제 부서 데이터와 구별합니다.

In [ ]:
patterns <- diag_patterns()
curve <- patterns$curve
c(sum_residual=sum(curve$residual),sum_x_residual=sum(curve$x*curve$residual))
plot(curve$fitted,curve$residual,pch=19,col="#087f8c",xlab="Fitted",ylab="Residual",main="Simulated curved mean, straight-line fit")
abline(h=0,lty=2)

### L4 · 예측–실행–설명

잔차합이0인데도 곡선이 남습니다. 이것이 모순인지 설명하고, R²나 p값보다 먼저 확인할 사항을 쓰세요.

In [ ]:
# answer <- "대수적 성질과 모형 가정의 차이: ..."

In [ ]:
stopifnot(nrow(res)==30,df.residual(fit)==27,max(abs(res$internal-rstandard(fit)))<1e-10,max(abs(res$external-rstudent(fit)))<1e-10,nrow(ans_summary)==4)